<a href="https://colab.research.google.com/github/ShreyasRamchandraDure/BIS_Lab/blob/main/GreyWolfOptimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Hospital departments
departments = ["Emergency", "ICU", "General"]

# Resource demand: Beds, Doctors, Nurses
demand = np.array([
    [45, 15, 25],
    [25, 10, 20],
    [50, 20, 35]
], dtype=float)

# Total available resources
available = np.array([100, 30, 60], dtype=float)

# Total demand for each resource
total_demand = demand.sum(axis=0)

# GWO parameters
wolves = 20
iterations = 100
dimensions = 9

lower = np.zeros(dimensions)
upper = demand.flatten()

# Fitness function
def fitness(position):
    allocation = position.reshape(3, 3)

    # Resource allocation error
    error = np.sum((demand - allocation) ** 2)

    # Penalty for exceeding available resources
    total = allocation.sum(axis=0)
    penalty = np.sum(np.maximum(total - available, 0) ** 2)

    return error + 1000 * penalty

# Initialize wolves
population = np.random.uniform(
    lower, upper, (wolves, dimensions)
)

alpha = np.zeros(dimensions)
beta = np.zeros(dimensions)
delta = np.zeros(dimensions)

alpha_score = beta_score = delta_score = float("inf")

# Grey Wolf Optimization
for t in range(iterations):

    for wolf in population:
        score = fitness(wolf)

        if score < alpha_score:
            delta_score = beta_score
            delta = beta.copy()

            beta_score = alpha_score
            beta = alpha.copy()

            alpha_score = score
            alpha = wolf.copy()

        elif score < beta_score:
            delta_score = beta_score
            delta = beta.copy()

            beta_score = score
            beta = wolf.copy()

        elif score < delta_score:
            delta_score = score
            delta = wolf.copy()

    a = 2 - 2 * t / iterations

    for i in range(wolves):
        A1 = 2 * a * np.random.rand(dimensions) - a
        C1 = 2 * np.random.rand(dimensions)
        D_alpha = abs(C1 * alpha - population[i])
        X1 = alpha - A1 * D_alpha

        A2 = 2 * a * np.random.rand(dimensions) - a
        C2 = 2 * np.random.rand(dimensions)
        D_beta = abs(C2 * beta - population[i])
        X2 = beta - A2 * D_beta

        A3 = 2 * a * np.random.rand(dimensions) - a
        C3 = 2 * np.random.rand(dimensions)
        D_delta = abs(C3 * delta - population[i])
        X3 = delta - A3 * D_delta

        population[i] = (X1 + X2 + X3) / 3

        # Keep allocation within demand bounds
        population[i] = np.clip(population[i], lower, upper)

# Display result
best = alpha.reshape(3, 3)

print("Optimized Hospital Resource Allocation")
print("----------------------------------")

for i in range(3):
    print(departments[i])
    print("Beds    :", round(best[i][0]))
    print("Doctors :", round(best[i][1]))
    print("Nurses  :", round(best[i][2]))
    print()

print("Total Resources Used:")
print("Beds    :", round(best[:, 0].sum()))
print("Doctors :", round(best[:, 1].sum()))
print("Nurses  :", round(best[:, 2].sum()))

print("\nBest Fitness Value:", round(alpha_score, 2))

Optimized Hospital Resource Allocation
----------------------------------
Emergency
Beds    : 38
Doctors : 10
Nurses  : 18

ICU
Beds    : 17
Doctors : 7
Nurses  : 12

General
Beds    : 45
Doctors : 12
Nurses  : 29

Total Resources Used:
Beds    : 100
Doctors : 30
Nurses  : 60

Best Fitness Value: 369.41
